# Design Archaeology 🏺⚡
### Stratigraphic provenance for netlists and floorplans — the full experiment in one notebook

**One-line pitch.** Chip designs are archaeological sites — layers of decisions made under
constraints that no longer exist. We borrow *formal methods* from archaeology (stratigraphy,
seriation, residuality theory) to detect **vestigial constraints** — design rules that no
longer cause anything — and to reconstruct the genealogy of design decisions from version
history alone.

Mature designs accumulate sediment: inherited IP blocks, ECO patches on patches, timing
exceptions written for a process node three generations dead, placement regions nobody
dares delete. Engineers know some constraints are load-bearing and some are fossils, but
the only way to find out is *relax it and re-run the flow* — too expensive to do for
hundreds of constraints. Archaeology has spent a century formalizing exactly this problem:

| Archaeology | Chip design |
|---|---|
| Stratum (deposition layer) | A revision / ECO / constraint-file edit |
| Harris matrix (DAG of which layers presuppose which) | DAG of which design decisions depend on which |
| Seriation (relative dating from artifact style) | Ordering edits from diff features when history is lost |
| Residual artifact (deposited long after its use-life) | Vestigial constraint (rule that constrains nothing anymore) |
| Use / reuse / residuality distinction | Active / repurposed / dead constraint |

The ML contribution is a **learned surrogate for the expensive relax-and-re-run experiment,
evaluated against a real interventional oracle**. The humanities contribution is *method,
not metaphor* — residuality theory supplies the label taxonomy and the evaluation frame.

**What this notebook does, end to end (CPU only, no internet, ~5–10 min):**

1. Builds a tiny but real **timing-driven place-and-route flow** (analytical placement,
   STA, congestion) whose constraints genuinely steer optimization.
2. **Grows synthetic designs through an edit grammar** — every edit is a stratum, renames
   manufacture fossils, and ground-truth provenance is free.
3. Runs the **interventional oracle**: ~2,500 full flow runs that relax every constraint at
   three levels and label it *active* or *vestigial* against a seed-noise band.
4. Trains **gradient-boosted trees on cheap parser features** to replace the oracle;
   baselines include the dead-referent linter it must beat.
5. **Deletion experiment**: deletes the model's fossils all at once and re-runs the flow.
6. **Seriation**: recovers deposition order from diff content alone (Kendall τ), under
   squashed-commit corruption.
7. **Harris matrices**: interventional presupposition edges, and a classifier that
   predicts them.

Along the way it renders the designs as die shots, as stratigraphic sections of an
excavation trench, and as Harris matrices — the site the way an archaeologist would draw it.

## 1 · The flow: a tiny analytical place-and-route engine

Everything downstream needs a *cheap but real* physical-design flow: constraints must
actually steer optimization so that relaxing them can change outcomes. This engine has:

- **Netlist**: hierarchical cells (`comb` logic, `reg` registers, `macro` blocks, ports),
  driver→sinks nets, feed-forward timing graph (registers cut paths).
- **STA**: topological arrival-time propagation with placement-dependent wire delay;
  endpoint slacks against the clock, plus `max_delay` / `false_path` exceptions.
- **Timing-driven analytical placement**: iterative star-model solves with net weights
  from criticality, density spreading, region clamps, keepout push-out.
- **QoR**: WNS, TNS, wirelength (HPWL), routing-congestion overflow, DRC-ish overlap count.

Constraint types (the artifacts we will excavate): `clock`, `max_delay`, `false_path`,
`region`, `keepout`.

In [ ]:
import copy, json, math, time, itertools, warnings, zlib
from dataclasses import dataclass, field
from collections import defaultdict
from pathlib import Path

import numpy as np
import scipy.sparse as sparse

warnings.filterwarnings("ignore")

OUT = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("./output")
OUT.mkdir(parents=True, exist_ok=True)
IMG = OUT / "images"
IMG.mkdir(exist_ok=True)

In [ ]:
@dataclass
class Cell:
    name: str
    kind: str          # 'comb' | 'reg' | 'macro' | 'in' | 'out'
    delay: float       # intrinsic delay
    size: float = 0.012  # nominal radius, macros bigger

@dataclass
class Net:
    name: str
    driver: str
    sinks: list

@dataclass
class Constraint:
    cid: str
    ctype: str         # 'clock' | 'max_delay' | 'false_path' | 'region' | 'keepout'
    args: dict
    deposit_idx: int = -1   # which edit deposited it (ground truth, never a feature directly)

@dataclass
class Design:
    name: str
    cells: dict = field(default_factory=dict)      # name -> Cell
    nets: dict = field(default_factory=dict)       # name -> Net
    constraints: dict = field(default_factory=dict)  # cid -> Constraint
    edit_log: list = field(default_factory=list)     # list of edit records (strata)
    truth: dict = field(default_factory=dict)        # cid -> provenance ground truth
    cell_birth: dict = field(default_factory=dict)   # cell -> deposit edit idx
    default_period: float = 1.0

    def clone(self):
        return copy.deepcopy(self)

    def referents(self, c: Constraint):
        """Names a constraint points at."""
        t, a = c.ctype, c.args
        if t == "clock":       return [a["port"]]
        if t == "max_delay":   return [a["src"], a["dst"]]
        if t == "false_path":  return [a["src"], a["dst"]]
        if t == "region":      return [a["prefix"]]
        if t == "keepout":     return []
        return []

    def referent_alive(self, c: Constraint):
        """Per-referent liveness. A region prefix is alive if any cell matches."""
        out = []
        for r in self.referents(c):
            if c.ctype == "region":
                out.append(any(n.startswith(r) for n in self.cells))
            else:
                out.append(r in self.cells)
        return out

In [ ]:
# ---------------- Static timing analysis ----------------

WIRE_DELAY = 0.55      # delay per unit manhattan distance
GRID = 16              # congestion grid
ROUTE_CAP = 3.0        # per-bin routing capacity
DENSITY_GRID = 12

def _topo_order(design, fanin, fanout):
    # 'in'/'reg' nodes launch fresh (arrival independent of fanin), so only
    # comb/out nodes impose dependency edges; the comb subgraph is a DAG by construction
    dep = {}
    for n, c in design.cells.items():
        dep[n] = len(fanin[n]) if c.kind in ("comb", "out") else 0
    q = [n for n, d0 in dep.items() if d0 == 0]
    order = []
    while q:
        n = q.pop()
        order.append(n)
        for s, _ in fanout[n]:
            if design.cells[s].kind in ("comb", "out"):
                dep[s] -= 1
                if dep[s] == 0:
                    q.append(s)
    return order

def build_graph(design):
    """Timing edges driver->sink per net; registers cut paths (reg is endpoint+startpoint)."""
    fanin, fanout = defaultdict(list), defaultdict(list)
    for net in design.nets.values():
        d = net.driver
        if d not in design.cells:
            continue
        for s in net.sinks:
            if s not in design.cells:
                continue
            fanout[d].append((s, net.name))
            fanin[s].append((d, net.name))
    return fanin, fanout

def live_constraints(design, skip=()):
    """Constraints the tool actually honors: all referents alive, not skipped."""
    out = []
    for c in design.constraints.values():
        if c.cid in skip:
            continue
        if all(design.referent_alive(c)):
            out.append(c)
    return out

def clock_period(design, skip=()):
    period = design.default_period
    best = -1
    for c in live_constraints(design, skip):
        if c.ctype == "clock" and c.deposit_idx > best:
            best, period = c.deposit_idx, c.args["period"]
    return period

def sta(design, pos, skip=()):
    """Arrival/required times at fixed placement. Returns timing report dict."""
    fanin, fanout = build_graph(design)
    order = _topo_order(design, fanin, fanout)
    cons = live_constraints(design, skip)
    period = clock_period(design, skip)
    P = pos  # dict name -> (x, y)

    def wire(a, b):
        (ax, ay), (bx, by) = P[a], P[b]
        return WIRE_DELAY * (abs(ax - bx) + abs(ay - by))

    def forward(masked=frozenset()):
        arr = {}
        for n in order:
            c = design.cells[n]
            if n in masked:
                arr[n] = -math.inf
                continue
            if c.kind in ("in", "reg"):
                arr[n] = c.delay   # launch point: upstream arrivals don't pass through
                continue
            ins = [arr[d] + wire(d, n) for d, _ in fanin[n] if arr.get(d, -math.inf) > -math.inf]
            arr[n] = (max(ins) if ins else 0.0) + c.delay
        return arr

    def ep_arrival(arr, n):
        """Data arrival at an endpoint's *input* (registers capture at input)."""
        if design.cells[n].kind == "out":
            a = arr.get(n, 0.0)
            return 0.0 if a == -math.inf else a
        ins = [arr[d] + wire(d, n) for d, _ in fanin[n] if arr.get(d, -math.inf) > -math.inf]
        return max(ins) if ins else 0.0

    arr = forward()

    # false_path masks, grouped by endpoint
    fp_by_dst = defaultdict(set)
    for c in cons:
        if c.ctype == "false_path":
            fp_by_dst[c.args["dst"]].add(c.args["src"])
    arr_fp = {}
    for dst, srcs in fp_by_dst.items():
        if dst in design.cells:
            arr_fp[dst] = ep_arrival(forward(masked=frozenset(srcs)), dst)

    # endpoint slacks vs clock
    endpoints = [n for n, c in design.cells.items()
                 if c.kind in ("reg", "out") and fanin[n]]
    ep_slack = {}
    for n in endpoints:
        a = arr_fp.get(n, ep_arrival(arr, n))
        ep_slack[n] = period - a

    # max_delay constraints: longest path src->dst
    md_slack = {}
    for c in cons:
        if c.ctype != "max_delay":
            continue
        src, dst = c.args["src"], c.args["dst"]
        d = {n: -math.inf for n in order}
        if src in d:
            d[src] = design.cells[src].delay
            for n in order:
                if d[n] == -math.inf:
                    continue
                if design.cells[n].kind == "reg" and n != src:
                    continue  # registers cut the path
                for s, _ in fanout[n]:
                    cand = d[n] + wire(n, s) + design.cells[s].delay
                    if cand > d[s]:
                        d[s] = cand
        pd = d.get(dst, -math.inf)
        md_slack[c.cid] = (c.args["value"] - pd) if pd > -math.inf else None

    slacks = list(ep_slack.values()) + [s for s in md_slack.values() if s is not None]
    wns = min(slacks) if slacks else period
    tns = sum(min(0.0, s) for s in slacks)

    # criticality per cell: worst downstream endpoint slack, normalized
    req = {n: math.inf for n in design.cells}
    for n in endpoints:
        req[n] = min(req[n], ep_slack[n])
    for c in cons:
        if c.ctype == "max_delay" and md_slack.get(c.cid) is not None:
            for e in (c.args["src"], c.args["dst"]):
                if e in req:
                    req[e] = min(req[e], md_slack[c.cid])
    for n in reversed(order):
        for s, _ in fanout[n]:
            if design.cells[s].kind not in ("reg",):
                req[n] = min(req[n], req[s])
            else:
                req[n] = min(req[n], ep_slack.get(s, math.inf))
    crit = {}
    for n in design.cells:
        s = req[n]
        crit[n] = 0.0 if s is math.inf else max(0.0, min(1.5, (period - s) / period - 0.35))
    return dict(period=period, arr=arr, ep_slack=ep_slack, md_slack=md_slack,
                wns=wns, tns=tns, crit=crit)

In [ ]:
# ---------------- Timing-driven analytical placement ----------------

def _fixed_positions(design, rng):
    """Ports pinned to chip boundary, deterministic per name."""
    fixed = {}
    ins = sorted(n for n, c in design.cells.items() if c.kind == "in")
    outs = sorted(n for n, c in design.cells.items() if c.kind == "out")
    for i, n in enumerate(ins):
        fixed[n] = (0.02, (i + 1) / (len(ins) + 1))
    for i, n in enumerate(outs):
        fixed[n] = (0.98, (i + 1) / (len(outs) + 1))
    return fixed

def place(design, seed=0, skip=(), timing_passes=3, sweeps=24):
    """Timing-driven star-model placement honoring live region/keepout constraints."""
    rng = np.random.default_rng(seed)
    names = list(design.cells)
    idx = {n: i for i, n in enumerate(names)}
    N = len(names)
    fixed = _fixed_positions(design, rng)
    fixed_mask = np.zeros(N, bool)
    # near-deterministic init (real tools are ~deterministic; seeds perturb slightly):
    # deterministic per-name spread + small seeded jitter
    h = np.array([[(zlib.crc32(n.encode()) % 997) / 997.0,
                   (zlib.crc32(n[::-1].encode()) % 991) / 991.0] for n in names])
    X = 0.30 + 0.40 * h + rng.normal(0, 0.015, size=(N, 2))
    for n, p in fixed.items():
        X[idx[n]] = p
        fixed_mask[idx[n]] = True

    # sparse incidence: nets x cells
    rows, cols = [], []
    net_names = list(design.nets)
    for r, nn in enumerate(net_names):
        net = design.nets[nn]
        for m in [net.driver] + net.sinks:
            if m in idx:
                rows.append(r)
                cols.append(idx[m])
    M = sparse.csr_matrix((np.ones(len(rows)), (rows, cols)), shape=(len(net_names), N))
    net_deg = np.asarray(M.sum(axis=1)).ravel().clip(1)
    cell_deg = np.asarray(M.sum(axis=0)).ravel().clip(1)

    cons = live_constraints(design, skip)
    regions = [c for c in cons if c.ctype == "region"]
    keepouts = [c for c in cons if c.ctype == "keepout"]
    region_members = []
    for c in regions:
        mem = np.array([i for n, i in idx.items() if n.startswith(c.args["prefix"])], int)
        region_members.append(mem)

    w = np.ones(len(net_names))
    pos = {n: tuple(X[idx[n]]) for n in names}
    for tp in range(timing_passes):
        rep = sta(design, pos, skip)
        crit = np.array([rep["crit"][n] for n in names])
        # net weight from driver criticality
        for r, nn in enumerate(net_names):
            d = design.nets[nn].driver
            w[r] = 1.0 + 3.0 * rep["crit"].get(d, 0.0)
        Wn = sparse.diags(w)
        for it in range(sweeps):
            cent = (Wn @ M @ X) / (w * net_deg)[:, None]     # weighted net centroids
            pull = M.T @ Wn @ cent
            wsum = np.asarray((M.T @ Wn @ np.ones((len(net_names), 1)))).ravel().clip(1e-9)
            tgt = pull / wsum[:, None]
            step = 0.45 if tp == 0 else 0.3
            X[~fixed_mask] = (1 - step) * X[~fixed_mask] + step * tgt[~fixed_mask]
            if it % 4 == 3:
                _spread(X, fixed_mask, rng)
            _apply_geo(X, fixed_mask, regions, region_members, keepouts)
        pos = {n: tuple(X[idx[n]]) for n in names}
    return pos

def _spread(X, fixed_mask, rng):
    g = DENSITY_GRID
    bins = np.clip((X * g).astype(int), 0, g - 1)
    flat = bins[:, 0] * g + bins[:, 1]
    count = np.bincount(flat, minlength=g * g).astype(float)
    cap = max(2.0, 2.2 * len(X) / (g * g))
    over = count[flat] > cap
    ctrs = (bins + 0.5) / g
    v = X - ctrs
    nrm = np.linalg.norm(v, axis=1, keepdims=True).clip(1e-6)
    push = 0.035 * v / nrm + rng.normal(0, 0.004, X.shape)
    m = over & ~fixed_mask
    X[m] = np.clip(X[m] + push[m], 0.01, 0.99)

def _apply_geo(X, fixed_mask, regions, region_members, keepouts):
    for c, mem in zip(regions, region_members):
        if len(mem) == 0:
            continue
        x0, y0, x1, y1 = c.args["rect"]
        X[mem, 0] = np.clip(X[mem, 0], x0, x1)
        X[mem, 1] = np.clip(X[mem, 1], y0, y1)
    for c in keepouts:
        x0, y0, x1, y1 = c.args["rect"]
        inside = (~fixed_mask & (X[:, 0] > x0) & (X[:, 0] < x1)
                  & (X[:, 1] > y0) & (X[:, 1] < y1))
        if not inside.any():
            continue
        ii = np.where(inside)[0]
        # push to nearest rect edge
        dl = X[ii, 0] - x0; dr = x1 - X[ii, 0]
        db = X[ii, 1] - y0; dt = y1 - X[ii, 1]
        d = np.stack([dl, dr, db, dt], 1)
        side = d.argmin(1)
        X[ii[side == 0], 0] = max(x0 - 0.005, 0.005)
        X[ii[side == 1], 0] = min(x1 + 0.005, 0.995)
        X[ii[side == 2], 1] = max(y0 - 0.005, 0.005)
        X[ii[side == 3], 1] = min(y1 + 0.005, 0.995)

In [ ]:
# ---------------- QoR ----------------

def qor(design, seed=0, skip=(), return_state=False):
    """Run the full flow (place + STA + congestion) and report QoR metrics."""
    pos = place(design, seed=seed, skip=skip)
    rep = sta(design, pos, skip)
    names = list(design.cells)
    P = np.array([pos[n] for n in names])

    wl = 0.0
    dem = np.zeros((GRID, GRID))
    for net in design.nets.values():
        pins = [pos[m] for m in [net.driver] + net.sinks if m in pos]
        if len(pins) < 2:
            continue
        xs = [p[0] for p in pins]; ys = [p[1] for p in pins]
        h = (max(xs) - min(xs)) + (max(ys) - min(ys))
        wl += h
        b0 = np.clip(int(min(xs) * GRID), 0, GRID - 1); b1 = np.clip(int(max(xs) * GRID), 0, GRID - 1)
        c0 = np.clip(int(min(ys) * GRID), 0, GRID - 1); c1 = np.clip(int(max(ys) * GRID), 0, GRID - 1)
        nb = (b1 - b0 + 1) * (c1 - c0 + 1)
        dem[b0:b1 + 1, c0:c1 + 1] += (h * GRID + 0.5) / nb
    cong = float(np.maximum(0.0, dem - ROUTE_CAP).sum())

    # DRC-ish: pairwise proximity violations via grid hashing
    sizes = np.array([design.cells[n].size for n in names])
    g = 20
    binid = np.clip((P * g).astype(int), 0, g - 1)
    buckets = defaultdict(list)
    for i, (a, b) in enumerate(binid):
        buckets[(a, b)].append(i)
    drc = 0
    for (a, b), members in buckets.items():
        cand = []
        for da in (-1, 0, 1):
            for db in (-1, 0, 1):
                cand += buckets.get((a + da, b + db), [])
        for i in members:
            for j in cand:
                if j <= i:
                    continue
                if np.abs(P[i] - P[j]).sum() < 0.6 * (sizes[i] + sizes[j]):
                    drc += 1
    m = dict(wns=rep["wns"], tns=rep["tns"], wl=wl, cong=cong, drc=float(drc))
    if return_state:
        return m, pos, rep, dem
    return m

def path_delay(design, pos, src, dst):
    """Longest src→dst delay at a placement (registers cut interior); None if no path."""
    fanin, fanout = build_graph(design)
    order = _topo_order(design, fanin, fanout)
    if src not in design.cells or dst not in design.cells:
        return None
    d = {n: -math.inf for n in order}
    d[src] = design.cells[src].delay
    for n in order:
        if d[n] == -math.inf or (design.cells[n].kind == "reg" and n != src):
            continue
        for s, _ in fanout[n]:
            (ax, ay), (bx, by) = pos[n], pos[s]
            cand = d[n] + WIRE_DELAY * (abs(ax - bx) + abs(ay - by)) + design.cells[s].delay
            if cand > d[s]:
                d[s] = cand
    pd = d.get(dst, -math.inf)
    return None if pd == -math.inf else pd

def reachable_endpoints(design, src):
    """Timing endpoints (reg/out) reachable from src without crossing a register."""
    _, fanout = build_graph(design)
    seen, stack, ends = {src}, [src], set()
    while stack:
        n = stack.pop()
        for s, _ in fanout[n]:
            if s in seen:
                continue
            seen.add(s)
            k = design.cells[s].kind
            if k in ("reg", "out"):
                ends.add(s)
            elif k == "comb":
                stack.append(s)
    return sorted(ends)

def arrival_dominator(design, pos, rep, dst):
    """Walk back from endpoint dst along the max-arrival fanin chain to its launch point."""
    fanin, _ = build_graph(design)
    arr = rep["arr"]
    def best_fanin(n):
        cands = [(arr.get(d, -math.inf) + WIRE_DELAY * (abs(pos[d][0] - pos[n][0])
                  + abs(pos[d][1] - pos[n][1])), d) for d, _ in fanin[n]]
        cands = [c for c in cands if c[0] > -math.inf]
        return max(cands)[1] if cands else None
    n = best_fanin(dst)
    while n is not None and design.cells[n].kind == "comb":
        n = best_fanin(n)
    return n

METRICS = ["wns", "tns", "wl", "cong", "drc"]
print("flow engine loaded · metrics:", METRICS)

## 2 · Synthetic strata: growing designs through an edit grammar

Each design is *grown* through a scripted sequence of 40–70 edits (strata). The grammar:

| edit | archaeological role |
|---|---|
| `add_module` | new occupation layer (cells + nets deposited) |
| `add_clock` / `add_max_delay` / `add_false_path` | artifacts deposited in a layer |
| `add_region` / `add_keepout` | architecture: walls and floors |
| `rename_hierarchy` | **the fossil factory** — constraints keep pointing at dead names |
| `eco_fix` | a repair intrusion cutting through earlier layers |
| `cosmetic` | taphonomic noise (reformat diffs, no semantic content) |

Because the sequence is scripted we know, for every constraint, **when it was deposited,
what it referenced then, and whether the referent still exists** — free ground truth.
The grammar deliberately manufactures *live-referent* vestigial constraints (a `max_delay`
superseded by a tighter one; a region larger than the whole die; a `false_path` on a path
that was never critical) so the dead-referent linter heuristic cannot win by default.

In [ ]:
MODULE_SIZES = (10, 26)

def _new_module(design, prefix, rng, edit_idx):
    """Deposit a cluster of cells with internal feed-forward logic + a few registers."""
    n = rng.integers(*MODULE_SIZES)
    names = []
    for i in range(n):
        kind = "reg" if rng.random() < 0.22 else "comb"
        delay = float(rng.uniform(0.02, 0.09)) if kind == "comb" else float(rng.uniform(0.03, 0.06))
        nm = f"{prefix}/c{i:03d}"
        design.cells[nm] = Cell(nm, kind, delay)
        design.cell_birth[nm] = edit_idx
        names.append(nm)
    if rng.random() < 0.35:
        nm = f"{prefix}/mac0"
        design.cells[nm] = Cell(nm, "macro", float(rng.uniform(0.05, 0.1)), size=0.05)
        design.cell_birth[nm] = edit_idx
        names.append(nm)
    # feed-forward internal nets (driver index < sink index → acyclic)
    for i in range(1, len(names)):
        k = int(rng.integers(1, 3))
        srcs = rng.choice(i, size=min(k, i), replace=False)
        for s in np.atleast_1d(srcs):
            nn = f"n_{prefix.replace('/', '_')}_{i}_{s}"
            design.nets.setdefault(nn, Net(nn, names[int(s)], []))
            design.nets[nn].sinks.append(names[i])
    return names

def _mod_rank(prefix):
    return int("".join(ch for ch in prefix if ch.isdigit()) or "0")

def _wire_modules(design, rng):
    """Connect modules. Comb drivers only wire 'forward' in module rank; register
    drivers may wire anywhere — keeps the combinational subgraph acyclic."""
    prefixes = sorted({n.split("/")[0] for n in design.cells
                       if design.cells[n].kind not in ("in", "out")})
    if len(prefixes) < 2:
        return
    a, b = rng.choice(prefixes, size=2, replace=False)
    if _mod_rank(a) >= _mod_rank(b):
        src = _pick_cell(design, a, rng, kinds=("reg",))
    else:
        src = _pick_cell(design, a, rng)
    dst = _pick_cell(design, b, rng)
    if src and dst and src != dst:
        nn = f"x_{src.replace('/', '_')}__{dst.replace('/', '_')}"
        if nn not in design.nets:
            design.nets[nn] = Net(nn, src, [dst])

def _pick_cell(design, prefix, rng, kinds=("comb", "reg")):
    cand = [n for n, c in design.cells.items()
            if n.startswith(prefix + "/") and c.kind in kinds]
    return str(rng.choice(cand)) if cand else None

def _endpoints(design):
    fanin, fanout = build_graph(design)
    starts = [n for n, c in design.cells.items() if c.kind in ("in", "reg") and fanout[n]]
    ends = [n for n, c in design.cells.items() if c.kind in ("reg", "out") and fanin[n]]
    return starts, ends

In [ ]:
def grow_design(name, seed, n_edits=55):
    """Grow one design; every edit is logged as a stratum with a diff record."""
    rng = np.random.default_rng(seed)
    d = Design(name=name, default_period=1.0)
    # primordial layer: ports
    for i in range(4):
        d.cells[f"pi{i}"] = Cell(f"pi{i}", "in", 0.02)
        d.cells[f"po{i}"] = Cell(f"po{i}", "out", 0.01)
    mod_counter = itertools.count()
    con_counter = itertools.count()
    renames = {}

    def log(kind, text, touched, cid=None):
        d.edit_log.append(dict(idx=len(d.edit_log), kind=kind, text=text,
                               touched=list(touched), cid=cid))

    def new_cid():
        return f"{name}_k{next(con_counter):03d}"

    # first strata: a couple of modules + the clock
    for _ in range(3):
        p = f"u{next(mod_counter):02d}"
        cells = _new_module(d, p, rng, len(d.edit_log))
        log("add_module", f"module {p} ({len(cells)} cells)", cells)
    # wire ports into fabric
    for i in range(4):
        tgt = _pick_cell(d, "u00", rng) or _pick_cell(d, "u01", rng)
        if tgt:
            d.nets[f"pin{i}"] = Net(f"pin{i}", f"pi{i}", [tgt])
        src = _pick_cell(d, "u01", rng) or _pick_cell(d, "u02", rng)
        if src:
            d.nets[f"pout{i}"] = Net(f"pout{i}", src, [f"po{i}"])
    cid = new_cid()
    period = float(rng.uniform(0.9, 1.15))
    d.constraints[cid] = Constraint(cid, "clock", dict(port="pi0", period=period), len(d.edit_log))
    d.truth[cid] = dict(deposit=len(d.edit_log), referents=["pi0"], intent="active")
    log("add_clock", f"create_clock -period {period:.3f} [get_ports pi0]", ["pi0"], cid)

    md_by_pair = {}   # (src,dst) -> (cid, value): supersession bookkeeping
    cache = {"pos": None, "rep": None, "dirty": True}

    def snapshot():
        """The 'engineer runs the flow' before writing a timing constraint."""
        if cache["dirty"]:
            cache["pos"] = place(d, seed=0)
            cache["rep"] = sta(d, cache["pos"])
            cache["dirty"] = False
        return cache["pos"], cache["rep"]

    while len(d.edit_log) < n_edits:
        r = rng.random()
        starts, ends = _endpoints(d)
        if r < 0.16:
            p = f"u{next(mod_counter):02d}"
            cells = _new_module(d, p, rng, len(d.edit_log))
            for _ in range(3):
                _wire_modules(d, rng)
            cache["dirty"] = True
            log("add_module", f"module {p} ({len(cells)} cells)", cells)
        elif r < 0.24:
            _wire_modules(d, rng)
            cache["dirty"] = True
            log("add_wire", "route new inter-module connection", [])
        elif r < 0.46 and starts:
            # max_delay on a real path — tight (active), loose (live fossil), or superseding
            pos, rep = snapshot()
            src = None
            for _ in range(6):
                cand = str(rng.choice(starts))
                ends_r = reachable_endpoints(d, cand)
                if ends_r:
                    src, dst = cand, str(rng.choice(ends_r))
                    break
            if src is None:
                continue
            pd = path_delay(d, pos, src, dst)
            if pd is None:
                continue
            flavor = rng.random()
            if flavor < 0.45:
                val, intent = float(pd * rng.uniform(0.45, 0.7)), "active"      # violated → binding
            elif flavor < 0.75:
                val, intent = float(pd * rng.uniform(2.5, 4.0)), "loose"        # live fossil
            else:
                val, intent = float(pd * rng.uniform(0.4, 0.6)), "active"
                old = md_by_pair.get((src, dst))
                if old and old[1] > val and old[0] in d.truth:
                    d.truth[old[0]]["intent"] = "superseded"                    # live fossil
            cid = new_cid()
            d.constraints[cid] = Constraint(cid, "max_delay",
                                            dict(src=src, dst=dst, value=val), len(d.edit_log))
            d.truth[cid] = dict(deposit=len(d.edit_log), referents=[src, dst], intent=intent)
            md_by_pair[(src, dst)] = (cid, val)
            cache["dirty"] = True   # timing weights shift the next placement
            log("add_max_delay", f"set_max_delay {val:.3f} -from {src} -to {dst}", [src, dst], cid)
        elif r < 0.54 and ends:
            # false_path — on the dominating launch of a tight endpoint (active),
            # or on a random reachable-but-uncritical path (live fossil)
            pos, rep = snapshot()
            tight = sorted(rep["ep_slack"], key=rep["ep_slack"].get)
            src = dst = None
            if rng.random() < 0.5 and tight:
                dst = tight[int(rng.integers(0, min(4, len(tight))))]
                src = arrival_dominator(d, pos, rep, dst)
            else:
                for _ in range(6):
                    cand = str(rng.choice(starts)) if starts else None
                    ends_r = reachable_endpoints(d, cand) if cand else []
                    if ends_r:
                        src, dst = cand, str(rng.choice(ends_r))
                        break
            if not src or not dst:
                continue
            cid = new_cid()
            d.constraints[cid] = Constraint(cid, "false_path", dict(src=src, dst=dst), len(d.edit_log))
            d.truth[cid] = dict(deposit=len(d.edit_log), referents=[src, dst], intent="unknown")
            cache["dirty"] = True
            log("add_false_path", f"set_false_path -from {src} -to {dst}", [src, dst], cid)
        elif r < 0.66:
            prefixes = sorted({p for p in {n.split("/")[0] for n in d.cells if "/" in n}
                               if sum(m.startswith(p + "/") for m in d.cells) >= 6})
            if prefixes:
                p = str(rng.choice(prefixes))
                if rng.random() < 0.62:
                    # tight region *displaced* from the module's natural centroid,
                    # so it demonstrably bends the placement
                    pos, _ = snapshot()
                    mem = [n for n in d.cells if n.startswith(p + "/")]
                    cen = np.mean([pos[m] for m in mem], axis=0)
                    th = rng.uniform(0, 2 * np.pi)
                    dist = rng.uniform(0.25, 0.45)
                    cx = float(np.clip(cen[0] + dist * np.cos(th), 0.12, 0.88))
                    cy = float(np.clip(cen[1] + dist * np.sin(th), 0.12, 0.88))
                    w = rng.uniform(0.07, 0.12)
                    rect = (max(0, cx - w), max(0, cy - w), min(1, cx + w), min(1, cy + w))
                    intent = "active"
                else:                     # covers ~whole die: live fossil
                    rect = (0.02, 0.02, 0.98, 0.98)
                    intent = "loose"
                cid = new_cid()
                d.constraints[cid] = Constraint(cid, "region",
                                                dict(prefix=p, rect=tuple(np.round(rect, 3))),
                                                len(d.edit_log))
                d.truth[cid] = dict(deposit=len(d.edit_log), referents=[p], intent=intent)
                cache["dirty"] = True
                log("add_region", f"create_region {p} " +
                    " ".join(f"{v:.3f}" for v in rect), [p], cid)
        elif r < 0.72:
            cx, cy = rng.uniform(0.1, 0.9, 2)
            w = rng.uniform(0.05, 0.14)
            rect = (max(0, cx - w), max(0, cy - w), min(1, cx + w), min(1, cy + w))
            cid = new_cid()
            d.constraints[cid] = Constraint(cid, "keepout", dict(rect=tuple(np.round(rect, 3))),
                                            len(d.edit_log))
            d.truth[cid] = dict(deposit=len(d.edit_log), referents=[], intent="unknown")
            cache["dirty"] = True
            log("add_keepout", "create_keepout " + " ".join(f"{v:.3f}" for v in rect), [], cid)
        elif r < 0.80:
            # rename_hierarchy: THE fossil factory — constraints are NOT updated
            prefixes = sorted({n.split("/")[0] for n in d.cells if "/" in n})
            fresh = [p for p in prefixes if p not in renames.values()]
            if fresh:
                p = str(rng.choice(fresh))
                np_ = p + "_r"
                mapping = {n: np_ + n[len(p):] for n in list(d.cells) if n.startswith(p + "/")}
                for old, new in mapping.items():
                    d.cells[new] = d.cells.pop(old)
                    d.cells[new].name = new
                    d.cell_birth[new] = d.cell_birth.pop(old, 0)
                for net in d.nets.values():
                    net.driver = mapping.get(net.driver, net.driver)
                    net.sinks = [mapping.get(s, s) for s in net.sinks]
                renames[p] = np_
                cache["dirty"] = True
                log("rename", f"rename hierarchy {p} -> {np_}", [p, np_])
        elif r < 0.88:
            # ECO: resize a slow cell (repair intrusion)
            combs = [n for n, c in d.cells.items() if c.kind == "comb"]
            if combs:
                n_ = str(rng.choice(combs))
                d.cells[n_].delay = max(0.01, d.cells[n_].delay * 0.6)
                cache["dirty"] = True
                log("eco_fix", f"eco: upsize {n_} (delay -40%)", [n_])
        else:
            log("cosmetic", "reformat constraint file (whitespace only)", [])

    # finalize ground truth liveness
    for cid, c in d.constraints.items():
        d.truth[cid]["alive"] = all(d.referent_alive(c))
    return d

print("grammar loaded")

## 3 · The interventional oracle

Ground truth is *interventional*, not textual. For every constraint we relax it at up to
three levels — **delete**, **loosen 50%**, **loosen 10%** — re-run the whole flow across
seeds, and measure ΔQoR against a per-design, per-metric **noise band** estimated from
seed variance. A constraint is **vestigial** iff no relaxation moves any metric out of
the noise band; the normalized max |ΔQoR|/band is its **effect size**.

For the Harris matrix we test sampled pairs (A deposited before B): edge **A→B** iff
relaxing A materially changes B's measured effect — B *presupposes* A.

In [ ]:
NOISE_FLOORS = dict(wns=0.012, tns=0.02, wl=0.45, cong=8.0, drc=40.0)
BAND_K = 2.5          # band = max(BAND_K * seed_std, floor)
VESTIGIAL_THRESH = 1.0
HARRIS_THRESH = 1.0

def flow_stats(design, seeds, skip=()):
    runs = [qor(design, seed=s, skip=skip) for s in seeds]
    return {m: float(np.mean([r[m] for r in runs])) for m in METRICS}, runs

def estimate_band(design, band_seeds, int_seeds):
    """Band = BAND_K sigmas of the Δ-of-means estimator: σ estimated from many
    baseline seeds, scaled by √(2/S) for S-seed intervention means."""
    _, runs = flow_stats(design, band_seeds)
    S = max(2, len(int_seeds))
    band = {}
    for m in METRICS:
        sd = float(np.std([r[m] for r in runs], ddof=1)) if len(runs) > 1 else 0.0
        band[m] = max(BAND_K * sd * math.sqrt(2.0 / S), NOISE_FLOORS[m])
    base = {m: float(np.mean([r[m] for r in runs[:len(int_seeds)]])) for m in METRICS}
    return base, band

def relax_levels(c):
    """(level_name, modified_args_or_None). None ⇒ delete (skip the constraint)."""
    lv = [("delete", None)]
    a = c.args
    if c.ctype == "max_delay":
        lv += [("loosen50", {**a, "value": a["value"] * 1.5}),
               ("loosen10", {**a, "value": a["value"] * 1.1})]
    elif c.ctype == "clock":
        lv += [("loosen50", {**a, "period": a["period"] * 1.5}),
               ("loosen10", {**a, "period": a["period"] * 1.1})]
    elif c.ctype in ("region", "keepout"):
        x0, y0, x1, y1 = a["rect"]
        cx, cy, hw, hh = (x0 + x1) / 2, (y0 + y1) / 2, (x1 - x0) / 2, (y1 - y0) / 2
        s50, s10 = (1.5, 1.1) if c.ctype == "region" else (0.5, 0.9)
        def rect(s):
            return (max(0, cx - hw * s), max(0, cy - hh * s),
                    min(1, cx + hw * s), min(1, cy + hh * s))
        lv += [("loosen50", {**a, "rect": rect(s50)}),
               ("loosen10", {**a, "rect": rect(s10)})]
    return lv  # false_path: delete only

def _modified(design, cid, args):
    d2 = design.clone()
    d2.constraints[cid].args = args
    return d2

def norm_delta(mod_mean, base_mean, band):
    return {m: (mod_mean[m] - base_mean[m]) / band[m] for m in METRICS}

def run_oracle(design, seeds=(0, 1, 2, 3), band_seeds=tuple(range(8)), verbose=True):
    """Label every constraint. Returns dict cid -> record, plus base/band and cache."""
    base, band = estimate_band(design, band_seeds, seeds)
    labels, delete_means = {}, {}
    for i, (cid, c) in enumerate(design.constraints.items()):
        per_level = {}
        for lname, args in relax_levels(c):
            if args is None:
                mod_mean, _ = flow_stats(design, seeds, skip=(cid,))
                delete_means[cid] = mod_mean
            else:
                mod_mean, _ = flow_stats(_modified(design, cid, args), seeds)
            nd = norm_delta(mod_mean, base, band)
            per_level[lname] = dict(delta=nd, z=max(abs(v) for v in nd.values()))
        eff = max(v["z"] for v in per_level.values())
        labels[cid] = dict(effect=eff, vestigial=eff < VESTIGIAL_THRESH,
                           per_level=per_level,
                           delete_delta=per_level["delete"]["delta"])
    n_v = sum(1 for r in labels.values() if r["vestigial"])
    if verbose:
        print(f"  {design.name}: {len(labels)} constraints, {n_v} vestigial "
              f"({100 * n_v / max(1, len(labels)):.0f}%)")
    return dict(base=base, band=band, labels=labels, delete_means=delete_means)

In [ ]:
def harris_ground_truth(design, oracle, seeds=(0, 1, 2, 3), max_pairs=30, rng=None):
    """Interventional pair tests: edge A→B iff B's effect changes when A is relaxed."""
    rng = rng or np.random.default_rng(7)
    cids = sorted(design.constraints, key=lambda c: design.constraints[c].deposit_idx)
    pairs = [(a, b) for i, a in enumerate(cids) for b in cids[i + 1:]]
    # bias sampling toward pairs that share referents or geometry (where edges live),
    # but keep a random slice so negatives are represented
    def touch(cid):
        c = design.constraints[cid]
        s = set()
        for r in design.referents(c):
            if c.ctype == "region":
                s |= {n for n in design.cells if n.startswith(r)}
            else:
                s.add(r)
        return s
    tsets = {cid: touch(cid) for cid in cids}
    scored = sorted(pairs, key=lambda p: -len(tsets[p[0]] & tsets[p[1]]))
    keep = scored[:max_pairs // 2]
    rest = [p for p in scored[max_pairs // 2:]]
    if rest:
        keep += [rest[i] for i in rng.choice(len(rest),
                 size=min(max_pairs - len(keep), len(rest)), replace=False)]
    base, band = oracle["base"], oracle["band"]
    out = {}
    for a, b in keep:
        eB = norm_delta(oracle["delete_means"][b], base, band)             # effect of B
        mean_a = oracle["delete_means"][a]
        mean_ab, _ = flow_stats(design, seeds, skip=(a, b))
        eB_given = {m: (mean_ab[m] - mean_a[m]) / band[m] for m in METRICS}  # effect of B | ¬A
        gap = max(abs(eB_given[m] - eB[m]) for m in METRICS)
        out[(a, b)] = dict(edge=gap > HARRIS_THRESH, gap=gap)
    return out

print("oracle loaded")

## 4 · Features: what a parser (and a tool log) can see without re-running the flow

Everything here is derivable from the final design state + constraint text + one baseline
flow run — **no interventions**. The oracle labels are never features.

- constraint type, arity
- **referent liveness** (dead referent ≈ residual artifact — this is the linter's whole toolkit)
- **age**: deposition position in the revision sequence; recency of referent edits
- **structural**: fanout of touched objects, referent-sharing overlap with other constraints
- **slack margin / geometric bindingness** at the baseline placement
- **static recheck**: re-run *STA only* (placement frozen) with the constraint dropped —
  the cheap "does the tool report it binding" signal; costs milliseconds, not a flow run.

In [ ]:
FEATURE_NAMES = [
    "t_clock", "t_max_delay", "t_false_path", "t_region", "t_keepout",
    "n_referents", "frac_alive", "any_dead",
    "age_frac", "referent_recency",
    "fanout", "overlap_cnt",
    "slack_margin", "geo_binding", "rect_area",
    "static_dwns", "static_dtns",
]

def _last_touch(design):
    lt = {}
    for e in design.edit_log:
        for t in e["touched"]:
            lt[t] = e["idx"]
    return lt

def extract_features(design, verbose=False):
    """Feature matrix + metadata for every constraint of one design."""
    mbase, pos, rep, dem = qor(design, seed=0, return_state=True)
    lt = _last_touch(design)
    T = max(1, len(design.edit_log))
    rows, meta = [], []
    for cid, c in design.constraints.items():
        alive = design.referent_alive(c)
        refs = design.referents(c)
        onehot = [float(c.ctype == t) for t in ("clock", "max_delay", "false_path", "region", "keepout")]
        frac_alive = float(np.mean(alive)) if alive else 1.0
        any_dead = float(not all(alive))
        age = c.deposit_idx / T
        rec = 0.0
        for r in refs:
            cands = [v for k, v in lt.items() if k == r or k.startswith(r + "/") or r.startswith(k + "/")]
            if cands:
                rec = max(rec, max(cands) / T)
        # fanout of referenced objects
        deg = defaultdict(int)
        for net in design.nets.values():
            deg[net.driver] += len(net.sinks)
            for s in net.sinks:
                deg[s] += 1
        if c.ctype == "region":
            members = [n for n in design.cells if n.startswith(c.args["prefix"])]
            fanout = float(len(members))
        else:
            fanout = float(sum(deg.get(r, 0) for r in refs))
        # overlap with other constraints (shared referents / overlapping rects)
        overlap = 0
        for cid2, c2 in design.constraints.items():
            if cid2 == cid:
                continue
            if set(refs) & set(design.referents(c2)):
                overlap += 1
            if c.ctype in ("region", "keepout") and c2.ctype in ("region", "keepout"):
                x0, y0, x1, y1 = c.args["rect"]; a0, b0, a1, b1 = c2.args["rect"]
                if not (x1 < a0 or a1 < x0 or y1 < b0 or b1 < y0):
                    overlap += 1
        # slack margin + geometric bindingness
        margin, geo, area = 5.0, 0.0, 0.0
        if c.ctype == "max_delay":
            s = rep["md_slack"].get(cid)
            margin = 5.0 if s is None else float(s)
        elif c.ctype == "clock":
            margin = float(min(rep["ep_slack"].values(), default=5.0))
        elif c.ctype == "false_path":
            src, dst = c.args["src"], c.args["dst"]
            pd = path_delay(design, pos, src, dst) if all(alive) else None
            margin = 5.0 if pd is None else float(rep["period"] - pd)
        elif c.ctype in ("region", "keepout"):
            x0, y0, x1, y1 = c.args["rect"]
            area = (x1 - x0) * (y1 - y0)
            if c.ctype == "region":
                members = [n for n in design.cells if n.startswith(c.args["prefix"])]
                if members:
                    P = np.array([pos[m] for m in members])
                    eps = 0.012
                    on_edge = ((np.abs(P[:, 0] - x0) < eps) | (np.abs(P[:, 0] - x1) < eps) |
                               (np.abs(P[:, 1] - y0) < eps) | (np.abs(P[:, 1] - y1) < eps))
                    geo = float(on_edge.mean())          # clamped cells ⇒ binding wall
                margin = float(area)
            else:
                b0 = np.clip(int(x0 * GRID), 0, GRID - 1); b1 = np.clip(int(x1 * GRID), 0, GRID - 1)
                c0 = np.clip(int(y0 * GRID), 0, GRID - 1); c1 = np.clip(int(y1 * GRID), 0, GRID - 1)
                geo = float(dem[b0:b1 + 1, c0:c1 + 1].mean())   # routing pressure at the wall
                margin = float(area)
        # static recheck: STA-only delta with the constraint dropped (placement frozen)
        rep2 = sta(design, pos, skip=(cid,))
        sd_wns, sd_tns = float(rep2["wns"] - rep["wns"]), float(rep2["tns"] - rep["tns"])
        rows.append(onehot + [float(len(refs)), frac_alive, any_dead, age, rec,
                              fanout, float(overlap), margin, geo, float(area),
                              sd_wns, sd_tns])
        meta.append(dict(design=design.name, cid=cid, ctype=c.ctype))
    return np.array(rows, float), meta

print("features loaded ·", len(FEATURE_NAMES), "features")

## 5 · Models and baselines

The workhorse is gradient-boosted trees on the cheap features (the headline number).
Baselines that must be beaten:
- **random**
- **linter**: flag constraints with dead referents — what a lint pass already does
- **age alone**: "old constraints are probably dead"
- **static recheck alone**: "the tool never reports it binding"

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier, HistGradientBoostingRegressor
from sklearn.metrics import roc_auc_score, average_precision_score, precision_recall_curve, roc_curve
from scipy.stats import spearmanr, kendalltau

def assemble_dataset(designs, oracles):
    X, y, eff, meta = [], [], [], []
    for d in designs:
        xi, mi = extract_features(d)
        orc = oracles[d.name]
        for row, m in zip(xi, mi):
            rec = orc["labels"][m["cid"]]
            X.append(row); y.append(int(rec["vestigial"])); eff.append(rec["effect"]); meta.append(m)
    return np.array(X), np.array(y), np.array(eff), meta

def fit_vestigial(X, y, seed=0):
    clf = HistGradientBoostingClassifier(max_depth=3, max_iter=220, learning_rate=0.08,
                                         min_samples_leaf=4, l2_regularization=0.5,
                                         random_state=seed)
    clf.fit(X, y)
    return clf

def fit_effect(X, eff, seed=0):
    reg = HistGradientBoostingRegressor(max_depth=3, max_iter=220, learning_rate=0.08,
                                        min_samples_leaf=4, l2_regularization=0.5,
                                        random_state=seed)
    reg.fit(X, np.log1p(eff))
    return reg

def baseline_scores(X, rng):
    """Vestigiality scores per baseline (higher = more vestigial)."""
    i = {n: j for j, n in enumerate(FEATURE_NAMES)}
    return {
        "random": rng.random(len(X)),
        "linter (dead referent)": X[:, i["any_dead"]],
        "age alone": X[:, i["age_frac"]],
        "static recheck alone": 1.0 / (1.0 + 50.0 * (np.abs(X[:, i["static_dwns"]])
                                                     + np.abs(X[:, i["static_dtns"]]))),
    }

def eval_scores(y, scores):
    out = {}
    for name, s in scores.items():
        try:
            out[name] = dict(auroc=roc_auc_score(y, s), ap=average_precision_score(y, s))
        except ValueError:
            out[name] = dict(auroc=float("nan"), ap=float("nan"))
    return out

print("models loaded")

## 6 · Seriation: relative dating when the history is lost

Given two edits with timestamps stripped, which came first? Archaeological seriation
orders artifacts by style; we order edits by *content*: an edit that references names
another edit introduced must come later; a constraint that names a hierarchy by its
**pre-rename** name must predate the rename. A pairwise "A before B?" classifier is
aggregated Borda/Bradley–Terry-style into a full deposition order, scored by Kendall's τ.
We also report robustness under **taphonomic corruption**: squashed commits.

In [ ]:
def _introduced(e):
    if e["kind"] == "add_module":
        return set(e["touched"])
    if e["kind"] == "rename":
        return {e["touched"][1]}
    return set()

def _referenced(e):
    if e["kind"] == "rename":
        return {e["touched"][0]}
    if e["kind"] == "add_module":
        return set()
    return set(e["touched"])

EDIT_KINDS = ["add_module", "add_wire", "add_clock", "add_max_delay", "add_false_path",
              "add_region", "add_keepout", "rename", "eco_fix", "cosmetic", "squashed"]

def _under(name, prefixes):
    return any(name == p or name.startswith(p + "/") for p in prefixes)

def _pair_dir_features(a, b):
    """Directional evidence that a came before b."""
    ia, ib = _introduced(a), _introduced(b)
    ra, rb = _referenced(a), _referenced(b)
    pa = {x.split("/")[0] for x in ia}
    f1 = float(any(_under(x, pa) or x in ia for x in rb))       # b references a's names
    f2 = float(b["kind"] == "rename" and any(_under(x, {b["touched"][0]}) for x in ra))
    f3 = float(len(ia)) / 30.0
    return [f1, f2, f3]

def _style(e):
    """Name-style features of a single edit — the analogue of ceramic style drift:
    newer strata use newer block names and post-rename ('_r') vocabulary."""
    names = list(_referenced(e)) + list(_introduced(e))
    prefixes = [x.split("/")[0] for x in names if "/" in x or x.startswith("u")]
    ranks = [_mod_rank(p) for p in prefixes if any(ch.isdigit() for ch in p)]
    max_rank = max(ranks) / 20.0 if ranks else -0.1
    mean_rank = float(np.mean(ranks)) / 20.0 if ranks else -0.1
    frac_renamed = float(np.mean(["_r" in x for x in names])) if names else 0.0
    return [max_rank, mean_rank, frac_renamed]

def seriation_features(a, b):
    ka = [float(a["kind"] == k) for k in EDIT_KINDS]
    kb = [float(b["kind"] == k) for k in EDIT_KINDS]
    return np.array(_pair_dir_features(a, b) + _pair_dir_features(b, a)
                    + _style(a) + _style(b) + ka + kb
                    + [len(a["text"]) / 80.0, len(b["text"]) / 80.0,
                       len(a["touched"]) / 30.0, len(b["touched"]) / 30.0])

def seriation_pairs(design, rng, max_pairs=1500):
    log = [e for e in design.edit_log]
    idxs = list(range(len(log)))
    pairs = list(itertools.combinations(idxs, 2))
    if len(pairs) > max_pairs:
        pairs = [pairs[i] for i in rng.choice(len(pairs), size=max_pairs, replace=False)]
    X, y = [], []
    for i, j in pairs:
        a, b = log[i], log[j]
        if rng.random() < 0.5:
            X.append(seriation_features(a, b)); y.append(1)   # a truly first
        else:
            X.append(seriation_features(b, a)); y.append(0)
    return np.array(X), np.array(y)

def corrupt_log(log, frac, rng):
    """Squash a fraction of edits into their successor (merged diff, union of touched)."""
    log = [dict(e) for e in log]
    n = len(log)
    kill = set(rng.choice(n - 1, size=int(frac * n), replace=False))
    out, pending = [], None
    for i, e in enumerate(log):
        if pending:
            e = dict(idx=e["idx"], kind="squashed", text=pending["text"] + " ; " + e["text"],
                     touched=pending["touched"] + e["touched"], cid=e.get("cid"))
            pending = None
        if i in kill:
            pending = e
        else:
            out.append(e)
    return out

def recover_order(log, clf, rng, max_pairs=4000):
    """Borda aggregation of pairwise win probabilities into a full order."""
    n = len(log)
    wins = np.zeros(n)
    pairs = list(itertools.combinations(range(n), 2))
    if len(pairs) > max_pairs:
        pairs = [pairs[i] for i in rng.choice(len(pairs), size=max_pairs, replace=False)]
    F = np.array([seriation_features(log[i], log[j]) for i, j in pairs])
    p = clf.predict_proba(F)[:, 1]     # P(i before j)
    for (i, j), pij in zip(pairs, p):
        wins[i] += pij
        wins[j] += 1 - pij
    order = np.argsort(-wins)          # earliest = most wins
    return order

def eval_seriation(designs_train, design_test, corruption=(0.0, 0.2, 0.4), seed=0):
    rng = np.random.default_rng(seed)
    Xs, ys = zip(*[seriation_pairs(d, rng) for d in designs_train])
    clf = HistGradientBoostingClassifier(max_depth=3, max_iter=150, random_state=seed)
    clf.fit(np.vstack(Xs), np.concatenate(ys))
    out = {}
    for q in corruption:
        log = corrupt_log(design_test.edit_log, q, rng) if q > 0 else design_test.edit_log
        Xt, yt = seriation_pairs(design_test, rng) if q == 0 else (None, None)
        acc = float(((clf.predict_proba(Xt)[:, 1] > 0.5) == yt).mean()) if q == 0 else None
        order = recover_order(log, clf, rng)
        true_rank = np.arange(len(log))
        tau = kendalltau(true_rank, np.argsort(order)).statistic
        out[q] = dict(pairwise_acc=acc, tau=float(tau))
    return out, clf

## 7 · Harris matrix: predicting the presupposition DAG

The interventional truth says edge **A→B** when relaxing A changes B's measured effect.
Can cheap features predict those edges? Pair features: shared referents, geometric
overlap, endpoint sharing (supersession chains), type pair, stratigraphic distance.

In [ ]:
def harris_pair_features(design, fa, fb, ca, cb):
    """fa/fb: feature rows; ca/cb: Constraint objects (A deposited before B)."""
    ra, rb = set(design.referents(ca)), set(design.referents(cb))
    shared = float(len(ra & rb))
    same_type = float(ca.ctype == cb.ctype)
    dep_gap = (cb.deposit_idx - ca.deposit_idx) / max(1, len(design.edit_log))
    rect_ov = 0.0
    if ca.ctype in ("region", "keepout") and cb.ctype in ("region", "keepout"):
        x0, y0, x1, y1 = ca.args["rect"]; a0, b0, a1, b1 = cb.args["rect"]
        rect_ov = max(0.0, min(x1, a1) - max(x0, a0)) * max(0.0, min(y1, b1) - max(y0, b0))
    ep_share = 0.0
    if ca.ctype in ("max_delay", "false_path") and cb.ctype in ("max_delay", "false_path"):
        ep_share = float(len({ca.args["src"], ca.args["dst"]}
                             & {cb.args["src"], cb.args["dst"]}))
    # region-of-endpoint coupling: timing constraint whose endpoints live in the region
    def in_region(reg, tc):
        if reg.ctype != "region" or tc.ctype not in ("max_delay", "false_path"):
            return 0.0
        return float(sum(str(tc.args[k]).startswith(reg.args["prefix"])
                         for k in ("src", "dst")))
    reg_couple = in_region(ca, cb) + in_region(cb, ca)
    return np.concatenate([fa, fb, [shared, same_type, dep_gap, rect_ov, ep_share, reg_couple]])

def harris_dataset(designs, oracles, harris_truths, feats_by_design):
    X, y, meta = [], [], []
    for d in designs:
        fmat, fmeta = feats_by_design[d.name]
        frow = {m["cid"]: fmat[i] for i, m in enumerate(fmeta)}
        for (a, b), rec in harris_truths[d.name].items():
            ca, cb = d.constraints[a], d.constraints[b]
            X.append(harris_pair_features(d, frow[a], frow[b], ca, cb))
            y.append(int(rec["edge"]))
            meta.append((d.name, a, b))
    return np.array(X), np.array(y), meta

print("seriation + harris loaded")

## 8 · The deletion experiment: teeth

Predictions are cheap; excavation permits are not. So we act on the model's word:
take every constraint the model calls vestigial on the **held-out design**, delete them
**all at once**, re-run the flow, and measure what broke. If the model is right, ΔQoR ≈ 0
and the constraint file just got k lines shorter. Compare against deleting a random set
of the same size, and against deleting what the dead-referent linter would delete.

In [ ]:
def deletion_experiment(design, oracle, clf, X, meta, thresh=0.8, seeds=(0, 1, 2, 3), seed=0):
    rng = np.random.default_rng(seed)
    base, band = oracle["base"], oracle["band"]
    p = clf.predict_proba(X)[:, 1]
    cids = [m["cid"] for m in meta]
    model_set = [c for c, pi in zip(cids, p) if pi > thresh]
    k = len(model_set)
    i_dead = FEATURE_NAMES.index("any_dead")
    linter_set = [c for c, row in zip(cids, X) if row[i_dead] > 0.5]
    rand_set = list(rng.choice(cids, size=min(k, len(cids)), replace=False))
    oracle_set = [c for c in cids if oracle["labels"][c]["vestigial"]]

    out = {}
    for name, ds in [("model top-k", model_set), ("random k", rand_set),
                     ("linter (dead refs)", linter_set), ("oracle (upper bound)", oracle_set)]:
        if not ds:
            out[name] = dict(k=0, harm=0.0, gain=0.0, delta={m: 0.0 for m in METRICS})
            continue
        mod, _ = flow_stats(design, seeds, skip=tuple(ds))
        nd = norm_delta(mod, base, band)
        # signed interpretation: wns/tns higher = better; wl/cong/drc lower = better
        harm = {m: (-nd[m] if m in ("wns", "tns") else nd[m]) for m in METRICS}
        out[name] = dict(k=len(ds), harm=max(0.0, max(harm.values())),
                         gain=max(0.0, -min(harm.values())), delta=nd)
    return out, model_set, p

print("deletion experiment loaded")

## 9 · Rendering the site

Die-shot floorplan renders (cells, macros, regions, keepouts, nets, critical path),
congestion heat, the design colored by *deposition stratum*, an excavation trench
section of the constraint record, and Harris matrices.

In [ ]:
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle, FancyBboxPatch, Polygon
from matplotlib.collections import LineCollection
from matplotlib.lines import Line2D
import matplotlib.cm as cm

plt.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 150, "font.size": 9.5,
    "font.family": "DejaVu Sans", "axes.titlesize": 11,
})

DIE_BG = "#0b0f14"
SOILS = ["#e8d5ae", "#d4b483", "#bc9163", "#a1744b", "#845c3b", "#68462c", "#4e3420", "#3a2617"]
LABEL_COLORS = {"active": "#39d98a", "vestigial-live": "#f2b134", "vestigial-dead": "#e0554d"}
TYPE_MARKERS = {"clock": "*", "max_delay": "o", "false_path": "^", "region": "s", "keepout": "D"}

def _module_prefix(n):
    return n.split("/")[0] if "/" in n else "ports"

def _module_colors(design):
    mods = sorted({_module_prefix(n) for n in design.cells})
    cmap = plt.get_cmap("tab20")
    return {m: (cmap(i % 20) if m != "ports" else (0.8, 0.8, 0.85, 1.0))
            for i, m in enumerate(mods)}

def render_floorplan(ax, design, pos, color_by="module", show_nets=True, rep=None,
                     title=None, net_alpha=0.06, show_geo=True):
    ax.set_facecolor(DIE_BG)
    ax.set_xlim(-0.02, 1.02); ax.set_ylim(-0.02, 1.02)
    ax.set_xticks([]); ax.set_yticks([]); ax.set_aspect("equal")
    for spine in ax.spines.values():
        spine.set_color("#39424e")
    # power-grid texture
    for g in np.linspace(0, 1, 17):
        ax.axvline(g, color="#161d26", lw=0.5, zorder=0)
        ax.axhline(g, color="#161d26", lw=0.5, zorder=0)
    # nets
    if show_nets:
        segs = []
        for net in design.nets.values():
            if net.driver not in pos:
                continue
            for s in net.sinks:
                if s in pos:
                    segs.append([pos[net.driver], pos[s]])
        if len(segs) > 900:
            idx = np.random.default_rng(0).choice(len(segs), 900, replace=False)
            segs = [segs[i] for i in idx]
        ax.add_collection(LineCollection(segs, colors="#7ec8ff", lw=0.4,
                                         alpha=net_alpha, zorder=1))
    # geometry constraints
    if show_geo:
        for c in live_constraints(design):
            if c.ctype == "region":
                x0, y0, x1, y1 = c.args["rect"]
                if (x1 - x0) * (y1 - y0) > 0.55:   # whole-die fossil: faint outline only
                    ax.add_patch(Rectangle((x0, y0), x1 - x0, y1 - y0, fill=False,
                                           edgecolor="#3d5a80", ls=":", lw=0.7,
                                           alpha=0.6, zorder=2))
                else:
                    ax.add_patch(Rectangle((x0, y0), x1 - x0, y1 - y0, fill=True,
                                           facecolor="#1f6feb", alpha=0.10, zorder=1))
                    ax.add_patch(Rectangle((x0, y0), x1 - x0, y1 - y0, fill=False,
                                           edgecolor="#58a6ff", ls="--", lw=1.1, zorder=6))
            elif c.ctype == "keepout":
                x0, y0, x1, y1 = c.args["rect"]
                ax.add_patch(Rectangle((x0, y0), x1 - x0, y1 - y0, fill=True,
                                       facecolor="#da3633", alpha=0.12, hatch="///",
                                       edgecolor="#f85149", lw=1.0, zorder=6))
    # cells
    mcolors = _module_colors(design)
    T = max(1, len(design.edit_log))
    strat_cmap = plt.get_cmap("magma")
    for n, cell in design.cells.items():
        x, y = pos[n]
        if color_by == "module":
            col = mcolors[_module_prefix(n)]
        else:
            col = strat_cmap(0.15 + 0.8 * design.cell_birth.get(n, 0) / T)
        if cell.kind == "macro":
            ax.add_patch(Rectangle((x - 0.028, y - 0.028), 0.056, 0.056,
                                   facecolor=col, edgecolor="#e6edf3", lw=0.9,
                                   alpha=0.95, zorder=5))
        elif cell.kind in ("in", "out"):
            ax.scatter([x], [y], marker=">" if cell.kind == "in" else "<",
                       s=34, c="#e6edf3", zorder=7, edgecolors="none")
        else:
            mk = "o" if cell.kind == "reg" else "s"
            ax.scatter([x], [y], marker=mk, s=13 if cell.kind == "reg" else 10,
                       c=[col], zorder=4,
                       edgecolors="#0b0f14", linewidths=0.2)
    # critical path glow
    if rep is not None:
        try:
            worst = min(rep["ep_slack"], key=rep["ep_slack"].get)
            fanin, _ = build_graph(design)
            path, node = [worst], worst
            for _ in range(60):
                cands = [(rep["arr"].get(d, -math.inf)
                          + WIRE_DELAY * (abs(pos[d][0] - pos[node][0])
                                          + abs(pos[d][1] - pos[node][1])), d)
                         for d, _ in fanin[node]]
                cands = [c for c in cands if c[0] > -math.inf]
                if not cands:
                    break
                node = max(cands)[1]
                path.append(node)
                if design.cells[node].kind in ("in", "reg"):
                    break
            pts = np.array([pos[n] for n in path])
            ax.plot(pts[:, 0], pts[:, 1], color="#ff2d55", lw=4.0, alpha=0.25, zorder=8)
            ax.plot(pts[:, 0], pts[:, 1], color="#ff5c77", lw=1.4, alpha=0.95, zorder=9,
                    marker="o", ms=2.5)
        except (ValueError, KeyError):
            pass
    if title:
        ax.set_title(title, color="#e6edf3" if ax.get_facecolor()[0] < 0.5 else "#111",
                     fontsize=10, pad=6)

def render_congestion(ax, dem, title="routing demand"):
    im = ax.imshow(dem.T, origin="lower", extent=(0, 1, 0, 1), cmap="inferno",
                   interpolation="bicubic")
    ax.set_xticks([]); ax.set_yticks([])
    ax.set_title(title, fontsize=10, pad=6, color="#e6edf3")
    return im

In [ ]:
def render_trench(design, oracle, fname=None, title=None):
    """Excavation trench section: strata bands, constraint artifacts, disturbances."""
    T = len(design.edit_log)
    fig, ax = plt.subplots(figsize=(11.5, 7.2))
    fig.patch.set_facecolor("#f4ead8")
    ax.set_facecolor("#f4ead8")
    ax.set_xlim(0, 10); ax.set_ylim(T + 2.5, -4.5)
    rng = np.random.default_rng(3)
    xs = np.linspace(0, 10, 240)
    epoch = 8
    n_bands = int(np.ceil(T / epoch)) + 1
    prev = np.full_like(xs, -1.2)
    for b in range(n_bands):
        y0 = b * epoch
        wig = 0.55 * np.sin(xs * (1.1 + 0.35 * b) + rng.uniform(0, 6)) \
            + rng.normal(0, 0.06, xs.shape).cumsum() * 0.06
        bot = np.minimum(y0 + epoch + wig, T + 2.5)
        ax.fill_between(xs, prev, bot, color=SOILS[b % len(SOILS)], zorder=1,
                        edgecolor="#3a2617", lw=0.5)
        prev = bot
    ax.fill_between(xs, -4.5, -1.2, color="#9fbf6b", zorder=1)   # topsoil / turf
    ax.axhline(-1.2, color="#3a2617", lw=1.2)
    depth = lambda idx: T - idx - 0.5      # edit 0 deposited first ⇒ deepest
    # disturbances: renames (full-width) and ECOs (narrow pits)
    n_ren = 0
    for e in design.edit_log:
        if e["kind"] == "rename":
            y = depth(e["idx"])
            ax.plot([0.15, 9.85], [y, y], color="#fffbe8", lw=2.6, alpha=0.75, zorder=3)
            ax.plot([0.15, 9.85], [y, y], color="#7a5230", lw=1.0, ls=(0, (6, 3)), zorder=4)
            xt, ha = (9.75, "right") if n_ren % 2 == 0 else (0.35, "left")
            ax.text(xt, y - 0.75, "⛏ " + e["text"].replace("rename hierarchy ", ""),
                    fontsize=6.8, ha=ha, color="#4a2f1d", style="italic", zorder=6)
            n_ren += 1
        elif e["kind"] == "eco_fix" and e["touched"]:
            tgt = e["touched"][0]
            birth = design.cell_birth.get(tgt, 0)
            xq = 0.6 + (zlib.crc32(tgt.encode()) % 880) / 100.0
            y0, y1 = depth(e["idx"]), depth(birth)   # cuts from its stratum down into older
            ax.add_patch(Polygon([[xq - 0.16, y0], [xq + 0.16, y0], [xq, y1]],
                                 closed=True, facecolor="#5f3f26", edgecolor="#2e1d10",
                                 lw=0.7, alpha=0.85, zorder=3))
    # constraint artifacts
    for cid, c in design.constraints.items():
        rec = oracle["labels"][cid]
        alive = all(design.referent_alive(c))
        lab = "active" if not rec["vestigial"] else ("vestigial-live" if alive else "vestigial-dead")
        y = depth(c.deposit_idx)
        x = 0.7 + (zlib.crc32(cid.encode()) % 860) / 100.0
        ax.scatter([x], [y], marker=TYPE_MARKERS[c.ctype],
                   s=200 if c.ctype == "clock" else 110,
                   facecolor=LABEL_COLORS[lab], edgecolor="#241708", lw=1.1, zorder=5)
        ax.text(x, y + 0.05, cid.split("_k")[-1], fontsize=5.6, ha="center", va="center",
                color="#241708", zorder=6, fontweight="bold")
    # depth scale (deeper = older)
    for k in range(0, T + 1, 10):
        y = T - k
        ax.plot([0.02, 0.14], [y, y], color="#3a2617", lw=1.2)
        ax.text(0.18, y + 0.15, f"edit {k}", fontsize=7, va="center", color="#3a2617")
    ax.set_xticks([])
    ax.set_yticks([])
    for spine in ax.spines.values():
        spine.set_visible(False)
    handles = [Line2D([], [], marker=TYPE_MARKERS[t], ls="", ms=8, markerfacecolor="#d8cbb2",
                      markeredgecolor="#241708", label=t) for t in TYPE_MARKERS]
    handles += [Line2D([], [], marker="o", ls="", ms=8, markerfacecolor=v,
                       markeredgecolor="#241708", label=k) for k, v in LABEL_COLORS.items()]
    handles += [Line2D([], [], color="#7a5230", ls=(0, (6, 3)), lw=1.4, label="rename disturbance"),
                Line2D([], [], marker="v", ls="", ms=9, markerfacecolor="#5f3f26",
                       markeredgecolor="#2e1d10", label="ECO intrusion")]
    ax.legend(handles=handles, loc="upper center", bbox_to_anchor=(0.5, -0.005),
              fontsize=7.6, ncol=6, framealpha=0.95, facecolor="#efe4cd",
              edgecolor="#7a5230", title="artifact typology · interventional label · disturbances",
              title_fontsize=7.8)
    ax.set_title(title or f"Trench section — design '{design.name}' "
                 f"(surface = latest edit; artifacts colored by interventional label)",
                 color="#2e1d10", fontsize=11.5)
    fig.tight_layout()
    if fname:
        fig.savefig(IMG / fname, bbox_inches="tight", facecolor=fig.get_facecolor())
    return fig

In [ ]:
def render_harris(design, truth_pairs, pred=None, fname=None, title=None):
    """Harris matrix: latest strata on top; a line drops from B to the A it presupposes."""
    cids = sorted({c for p in truth_pairs for c in p},
                  key=lambda c: design.constraints[c].deposit_idx)
    T = max(1, len(design.edit_log))
    epoch = 8
    rows = defaultdict(list)
    for c in cids:
        rows[design.constraints[c].deposit_idx // epoch].append(c)
    coord = {}
    for r, members in rows.items():
        for i, c in enumerate(sorted(members, key=lambda c: design.constraints[c].deposit_idx)):
            coord[c] = ((i + 1) / (len(members) + 1), r)
    fig, ax = plt.subplots(figsize=(10.5, 6.4))
    fig.patch.set_facecolor("#fbf7ef")
    ax.set_facecolor("#fbf7ef")
    tcmap = {"clock": "#8250df", "max_delay": "#0969da", "false_path": "#57606a",
             "region": "#1a7f37", "keepout": "#cf222e"}
    def elbow(a, b, color, ls, lw, z):
        (xa, ya), (xb, yb) = coord[a], coord[b]
        ym = (ya + yb) / 2
        ax.plot([xb, xb, xa, xa], [yb - 0.07, ym, ym, ya + 0.07],
                color=color, ls=ls, lw=lw, zorder=z, solid_capstyle="round")
    drawn = set()
    for (a, b), rec in truth_pairs.items():
        p = pred.get((a, b)) if pred else None
        if rec["edge"] and (p is None or p):
            elbow(a, b, "#1a7f37", "-", 2.2, 3)
        elif rec["edge"]:
            elbow(a, b, "#d4a72c", (0, (4, 2)), 2.0, 3)     # missed
        elif p:
            elbow(a, b, "#cf222e", (0, (1, 2)), 1.8, 2)     # false positive
        drawn.add((a, b))
    for c, (x, y) in coord.items():
        ct = design.constraints[c].ctype
        ax.add_patch(FancyBboxPatch((x - 0.052, y - 0.10), 0.104, 0.20,
                                    boxstyle="round,pad=0.012",
                                    facecolor="white", edgecolor=tcmap[ct], lw=1.6, zorder=5))
        ax.text(x, y + 0.025, c.split("_")[-1], ha="center", va="center", fontsize=8,
                fontweight="bold", zorder=6, color="#1f2328")
        ax.text(x, y - 0.055, ct, ha="center", va="center", fontsize=5.6, zorder=6,
                color=tcmap[ct])
    rmax = max(rows)
    for r in rows:
        ax.text(-0.035, r, f"strata\n{r * epoch}–{min(T, (r + 1) * epoch) - 1}",
                ha="center", va="center", fontsize=7, color="#57606a")
    ax.set_xlim(-0.09, 1.03)
    ax.set_ylim(-0.45, rmax + 0.45)
    ax.axis("off")
    handles = [Line2D([], [], color="#1a7f37", lw=2.2, label="edge (true ∧ predicted)"),
               Line2D([], [], color="#d4a72c", ls=(0, (4, 2)), lw=2.0, label="edge missed"),
               Line2D([], [], color="#cf222e", ls=(0, (1, 2)), lw=1.8, label="false edge")]
    ax.legend(handles=handles, loc="lower right", fontsize=8, framealpha=0.95)
    ax.set_title(title or f"Harris matrix — design '{design.name}' "
                 "(B ⇢ the A it presupposes; later strata on top)", fontsize=11)
    fig.tight_layout()
    if fname:
        fig.savefig(IMG / fname, bbox_inches="tight", facecolor=fig.get_facecolor())
    return fig

In [ ]:
def save_growth_gif(name, seed, n_edits, fname="growth.gif", step=4):
    """Animated die shot of the site accumulating strata."""
    from PIL import Image
    frames = []
    ks = list(range(6, n_edits + 1, step))
    if ks[-1] != n_edits:
        ks.append(n_edits)
    for k in ks:
        dk = grow_design(name, seed=seed, n_edits=k)
        pos = place(dk, seed=0)
        fig, ax = plt.subplots(figsize=(5.6, 5.6))
        fig.patch.set_facecolor(DIE_BG)
        render_floorplan(ax, dk, pos, color_by="stratum",
                         title=f"design '{name}' — edit {k}/{n_edits}")
        fig.tight_layout()
        fig.canvas.draw()
        img = np.asarray(fig.canvas.buffer_rgba())[..., :3]
        frames.append(Image.fromarray(img))
        plt.close(fig)
    frames[0].save(IMG / fname, save_all=True, append_images=frames[1:],
                   duration=420, loop=0)
    return IMG / fname

print("rendering loaded")

## 10 · The dig: generate the sites, run the oracle

6 training designs + 1 held-out design grown from a disjoint grammar seed.
The oracle is the compute hog (a few thousand full flow runs) — several minutes on CPU.

In [ ]:
RUN_T0 = time.time()
N_TRAIN = 6
N_EDITS = 65
HELD_EDITS = 75
ORACLE_SEEDS = (0, 1, 2, 3)
BAND_SEEDS = tuple(range(8))
HARRIS_PAIRS = 48
RESULTS = {}

print("growing designs …")
designs = [grow_design(f"d{i}", seed=101 + i, n_edits=N_EDITS) for i in range(N_TRAIN)]
held = grow_design("held", seed=777, n_edits=HELD_EDITS)
all_designs = designs + [held]
for d in all_designs:
    print(f"  {d.name}: {len(d.cells)} cells, {len(d.nets)} nets, "
          f"{len(d.constraints)} constraints, {len(d.edit_log)} strata")

print("\nrunning interventional oracle (relax → re-place-and-route → measure) …")
t0 = time.time()
oracles = {}
n_runs = 0
for d in all_designs:
    oracles[d.name] = run_oracle(d, seeds=ORACLE_SEEDS, band_seeds=BAND_SEEDS)
    n_runs += len(BAND_SEEDS) + sum(len(relax_levels(c)) for c in d.constraints.values()) * len(ORACLE_SEEDS)
print(f"  ≈{n_runs} full flow runs in {time.time() - t0:.0f}s")

# label composition vs provenance ground truth
comp = {}
for d in all_designs:
    orc = oracles[d.name]
    c = dict(active=0, vest_live=0, vest_dead=0)
    for cid, rec in orc["labels"].items():
        alive = all(d.referent_alive(d.constraints[cid]))
        if not rec["vestigial"]:
            c["active"] += 1
        elif alive:
            c["vest_live"] += 1
        else:
            c["vest_dead"] += 1
    comp[d.name] = c
tot_v = sum(c["vest_live"] + c["vest_dead"] for c in comp.values())
frac_live = sum(c["vest_live"] for c in comp.values()) / max(1, tot_v)
RESULTS["vestigial_frac"] = float(np.mean([(c["vest_live"] + c["vest_dead"])
                                           / max(1, sum(c.values())) for c in comp.values()]))
RESULTS["live_fossil_share"] = frac_live
print(f"\nvestigial constraints that a dead-referent linter CANNOT see: {100 * frac_live:.0f}%")

### The site, in pictures
The held-out design as a die shot (modules, regions, keepouts, nets, critical path in red),
the same placement colored by **deposition stratum** — the design as an archaeological
site — and its routing-demand heat.

In [ ]:
mQ, POS, REP, DEM = qor(held, seed=0, return_state=True)
fig, axes = plt.subplots(1, 3, figsize=(16.5, 5.8))
fig.patch.set_facecolor(DIE_BG)
render_floorplan(axes[0], held, POS, color_by="module", rep=REP,
                 title=f"'{held.name}' — floorplan by module · critical path")
render_floorplan(axes[1], held, POS, color_by="stratum", show_nets=False,
                 title="colored by deposition stratum (dark = ancient)")
render_congestion(axes[2], DEM, title="routing demand")
fig.tight_layout()
fig.savefig(IMG / "floorplan_trio.png", bbox_inches="tight", facecolor=DIE_BG)
plt.show()

In [ ]:
# the site accumulating strata — four excavation snapshots + an animation
snaps = [18, 34, 52, HELD_EDITS]
fig, axes = plt.subplots(1, 4, figsize=(17, 4.6))
fig.patch.set_facecolor(DIE_BG)
for ax, k in zip(axes, snaps):
    dk = grow_design("held", seed=777, n_edits=k)
    render_floorplan(ax, dk, place(dk, seed=0), color_by="stratum", net_alpha=0.05,
                     title=f"after edit {k}")
fig.suptitle("one design, four moments — strata accumulate", color="#e6edf3", y=1.02)
fig.tight_layout()
fig.savefig(IMG / "growth_montage.png", bbox_inches="tight", facecolor=DIE_BG)
plt.show()

gif_path = save_growth_gif("held", 777, HELD_EDITS, fname="growth.gif", step=4)
from IPython.display import Image as IPImage, display
display(IPImage(filename=str(gif_path)))

### Trench section
The full excavation record of the held-out design: strata, disturbances (renames, ECO
intrusions), and every constraint artifact colored by its **interventional** label.
Amber artifacts are the interesting ones: alive to every linter, dead to the flow.

In [ ]:
fig = render_trench(held, oracles["held"], fname="trench_held.png")
plt.show()

In [ ]:
# composition + effect sizes
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.2), width_ratios=[1, 1.35])
names = [d.name for d in all_designs]
b0 = [comp[n]["active"] for n in names]
b1 = [comp[n]["vest_live"] for n in names]
b2 = [comp[n]["vest_dead"] for n in names]
ax1.bar(names, b0, color=LABEL_COLORS["active"], label="active")
ax1.bar(names, b1, bottom=b0, color=LABEL_COLORS["vestigial-live"], label="vestigial (live referents)")
ax1.bar(names, b2, bottom=np.array(b0) + np.array(b1), color=LABEL_COLORS["vestigial-dead"],
        label="vestigial (dead referents)")
ax1.set_title("oracle label composition per design")
ax1.set_ylabel("constraints")
ax1.legend(fontsize=8)
rngj = np.random.default_rng(0)
types = ["clock", "max_delay", "false_path", "region", "keepout"]
for d in all_designs:
    for cid, rec in oracles[d.name]["labels"].items():
        ct = d.constraints[cid].ctype
        y = types.index(ct) + rngj.uniform(-0.22, 0.22)
        alive = all(d.referent_alive(d.constraints[cid]))
        lab = "active" if not rec["vestigial"] else ("vestigial-live" if alive else "vestigial-dead")
        ax2.scatter(max(rec["effect"], 0.02), y, s=26, color=LABEL_COLORS[lab],
                    edgecolor="#333", lw=0.4, alpha=0.85)
ax2.set_xscale("log")
ax2.axvline(VESTIGIAL_THRESH, color="#555", ls="--", lw=1)
ax2.text(VESTIGIAL_THRESH * 1.1, 4.45, "noise band", fontsize=8, color="#555")
ax2.set_yticks(range(len(types)), types)
ax2.set_xlabel("interventional effect size  (max |ΔQoR| / noise band, log scale)")
ax2.set_title("every constraint, by type — measured effect of relaxing it")
fig.tight_layout()
fig.savefig(IMG / "labels_effects.png", bbox_inches="tight")
plt.show()

## 11 · Can cheap features replace the oracle?
Leave-one-design-out CV on the training designs, then the held-out design.

In [ ]:
Xtr, ytr, efftr, mtr = assemble_dataset(designs, oracles)
Xte, yte, effte, mte = assemble_dataset([held], oracles)
print(f"train: {Xtr.shape[0]} constraints × {Xtr.shape[1]} features | held-out: {Xte.shape[0]}")

dn = np.array([m["design"] for m in mtr])
cv_scores, cv_y = [], []
for dname in sorted(set(dn)):
    tr, te = dn != dname, dn == dname
    cv_scores.append(fit_vestigial(Xtr[tr], ytr[tr]).predict_proba(Xtr[te])[:, 1])
    cv_y.append(ytr[te])
cv_scores, cv_y = np.concatenate(cv_scores), np.concatenate(cv_y)

clf = fit_vestigial(Xtr, ytr)
reg = fit_effect(Xtr, efftr)
p_te = clf.predict_proba(Xte)[:, 1]

ev_cv = eval_scores(cv_y, {"GBT (ours)": cv_scores,
                           **baseline_scores(Xtr, np.random.default_rng(0))})
ev_te = eval_scores(yte, {"GBT (ours)": p_te,
                          **baseline_scores(Xte, np.random.default_rng(1))})
print("\n=== vestigiality detection (AUROC / AP) ===")
print(f"{'method':28s} {'LODO-CV':>16s} {'held-out design':>16s}")
for k in ev_cv:
    print(f"{k:28s} {ev_cv[k]['auroc']:7.3f}/{ev_cv[k]['ap']:.3f} "
          f"{ev_te[k]['auroc']:8.3f}/{ev_te[k]['ap']:.3f}")
RESULTS["auroc_cv"], RESULTS["auroc_held"] = ev_cv["GBT (ours)"]["auroc"], ev_te["GBT (ours)"]["auroc"]
RESULTS["auroc_linter_held"] = ev_te["linter (dead referent)"]["auroc"]

pe_te = np.expm1(reg.predict(Xte))
rho = spearmanr(pe_te, effte).statistic
RESULTS["effect_spearman"] = float(rho)
print(f"\neffect-size regression, held-out Spearman ρ = {rho:.3f}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15.5, 4.4))
colors = {"GBT (ours)": "#d33682", "random": "#aaaaaa", "linter (dead referent)": "#268bd2",
          "age alone": "#859900", "static recheck alone": "#b58900"}
for name, s in {"GBT (ours)": p_te, **baseline_scores(Xte, np.random.default_rng(1))}.items():
    fpr, tpr, _ = roc_curve(yte, s)
    axes[0].plot(fpr, tpr, color=colors[name], lw=2.2 if "ours" in name else 1.4,
                 label=f"{name} ({roc_auc_score(yte, s):.2f})")
    pr, rc, _ = precision_recall_curve(yte, s)
    axes[1].plot(rc, pr, color=colors[name], lw=2.2 if "ours" in name else 1.4,
                 label=f"{name} ({average_precision_score(yte, s):.2f})")
axes[0].plot([0, 1], [0, 1], color="#ddd", ls="--", lw=1)
axes[0].set_title("ROC — held-out design"); axes[0].set_xlabel("FPR"); axes[0].set_ylabel("TPR")
axes[0].legend(fontsize=7.5, loc="lower right")
axes[1].set_title("precision–recall — held-out design")
axes[1].set_xlabel("recall"); axes[1].set_ylabel("precision")
axes[1].legend(fontsize=7.5, loc="lower left")
lab_cols = []
for m, e_true in zip(mte, effte):
    rec = oracles["held"]["labels"][m["cid"]]
    alive = all(held.referent_alive(held.constraints[m["cid"]]))
    lab_cols.append(LABEL_COLORS["active" if not rec["vestigial"]
                                 else ("vestigial-live" if alive else "vestigial-dead")])
axes[2].scatter(np.maximum(effte, 0.02), np.maximum(pe_te, 0.02), c=lab_cols,
                s=34, edgecolor="#333", lw=0.4)
axes[2].set_xscale("log"); axes[2].set_yscale("log")
axes[2].axvline(1, color="#999", ls="--", lw=0.8); axes[2].axhline(1, color="#999", ls="--", lw=0.8)
axes[2].set_xlabel("measured effect (oracle)"); axes[2].set_ylabel("predicted effect")
axes[2].set_title(f"effect-size calibration (ρ = {rho:.2f})")
fig.tight_layout()
fig.savefig(IMG / "detection_curves.png", bbox_inches="tight")
plt.show()

In [ ]:
from sklearn.inspection import permutation_importance
imp = permutation_importance(clf, Xte, yte, scoring="roc_auc", n_repeats=12, random_state=0)
order = np.argsort(imp.importances_mean)
fig, ax = plt.subplots(figsize=(8.5, 4.6))
ax.barh([FEATURE_NAMES[i] for i in order], imp.importances_mean[order],
        xerr=imp.importances_std[order], color="#d33682", alpha=0.85)
ax.set_title("permutation importance (held-out AUROC)")
ax.set_xlabel("Δ AUROC when feature is shuffled")
fig.tight_layout()
fig.savefig(IMG / "feature_importance.png", bbox_inches="tight")
plt.show()

## 12 · The deletion experiment
Act on the model's word: delete its confident vestigial calls **all at once**, re-run the
flow. Compare deleting a random set of the same size, the linter's dead-referent set, and
the oracle's own vestigial set (the upper bound — which is *not* zero: individually-
harmless artifacts interact, which is exactly why the Harris matrix exists).

In [ ]:
del_res, model_set, _ = deletion_experiment(held, oracles["held"], clf, Xte, mte, thresh=0.8)
print(f"{'deleted set':24s} {'k':>3s} {'harm':>6s} {'gain':>6s}  per-metric Δ (band units, + = metric ↑)")
for k, v in del_res.items():
    print(f"{k:24s} {v['k']:3d} {v['harm']:6.2f} {v['gain']:6.2f}  "
          + "  ".join(f"{m}={v['delta'][m]:+.2f}" for m in METRICS))
RESULTS["deletion"] = {k: dict(k=v["k"], harm=v["harm"], gain=v["gain"]) for k, v in del_res.items()}

fig, (axb, ax1, ax2) = plt.subplots(1, 3, figsize=(16.5, 5.2),
                                    width_ratios=[0.9, 1.05, 1.05])
sets = list(del_res)
w = 0.38
xs_ = np.arange(len(sets))
axb.bar(xs_ - w / 2, [del_res[k]["harm"] for k in sets], width=w,
        color="#cf222e", label="worst QoR harm")
axb.bar(xs_ + w / 2, [del_res[k]["gain"] for k in sets], width=w,
        color="#2aa198", label="best QoR gain")
axb.axhline(1.0, color="#555", ls="--", lw=1)
axb.text(2.45, 1.06, "noise band", fontsize=8, color="#555")
axb.set_xticks(xs_, [f"{k}\n(k={del_res[k]['k']})" for k in sets], fontsize=7.5)
axb.set_ylabel("ΔQoR in noise-band units")
axb.legend(fontsize=8)
axb.set_title("delete the set, re-run the flow")
fig.patch.set_facecolor("white")
for ax in (ax1, ax2):
    ax.set_facecolor(DIE_BG)
pos_after = place(held, seed=0, skip=tuple(model_set))
render_floorplan(ax1, held, POS, color_by="module", show_nets=False,
                 title=f"before — all {len(held.constraints)} constraints")
held_after = held.clone()
for c in model_set:
    del held_after.constraints[c]
render_floorplan(ax2, held_after, pos_after, color_by="module", show_nets=False,
                 title=f"after deleting the model's {len(model_set)} fossils")
for ax in (ax1, ax2):
    for spine in ax.spines.values():
        spine.set_color("#888")
for ax in (ax1, ax2):
    ax.title.set_color("#111")
fig.tight_layout()
fig.savefig(IMG / "deletion_experiment.png", bbox_inches="tight")
plt.show()

## 13 · Seriation: dating the strata from content alone
Timestamps stripped; a pairwise "which edit came first?" model, aggregated into a full
order (Borda), scored with Kendall's τ — including under squashed-commit corruption.

In [ ]:
ser_res, ser_clf = eval_seriation(designs, held, corruption=(0.0, 0.2, 0.4))
print("seriation on held-out design:")
for q, v in ser_res.items():
    acc = f"pairwise acc {v['pairwise_acc']:.2f}, " if v["pairwise_acc"] else ""
    print(f"  {int(q * 100):2d}% commits squashed: {acc}Kendall τ = {v['tau']:.2f}")
RESULTS["seriation"] = {str(q): v["tau"] for q, v in ser_res.items()}

fig, ax = plt.subplots(figsize=(6.8, 3.8))
qs = sorted(ser_res)
ax.bar([f"{int(q * 100)}%" for q in qs], [ser_res[q]["tau"] for q in qs],
       color=["#2aa198", "#b58900", "#cb4b16"], width=0.55)
ax.axhline(0, color="#333", lw=0.8)
ax.set_ylim(0, 1)
ax.set_ylabel("Kendall τ (recovered vs true deposition order)")
ax.set_xlabel("fraction of commits squashed (taphonomic corruption)")
ax.set_title(f"seriation of design '{held.name}'")
fig.tight_layout()
fig.savefig(IMG / "seriation.png", bbox_inches="tight")
plt.show()

## 14 · The Harris matrix
Interventional pair tests give the true presupposition DAG on sampled pairs; a pair
classifier on cheap features tries to reproduce it.

In [ ]:
print("running interventional pair tests (this is the second compute hog) …")
t0 = time.time()
htruth = {d.name: harris_ground_truth(d, oracles[d.name], max_pairs=HARRIS_PAIRS)
          for d in all_designs}
print(f"  {sum(len(v) for v in htruth.values())} pairs tested in {time.time() - t0:.0f}s; "
      f"edges found: " + ", ".join(f"{d.name}:{sum(r['edge'] for r in htruth[d.name].values())}"
                                   for d in all_designs))
feats_by_design = {d.name: extract_features(d) for d in all_designs}
Xh, yh, mh = harris_dataset(designs, oracles, htruth, feats_by_design)
Xht, yht, mht = harris_dataset([held], oracles, htruth, feats_by_design)
hclf = HistGradientBoostingClassifier(max_depth=2, max_iter=120, random_state=0)
hclf.fit(Xh, yh)
ph = hclf.predict_proba(Xht)[:, 1]
pred_pairs = {(a, b): bool(pv > 0.5) for (_, a, b), pv in zip(mht, ph)}
if yht.sum() > 0:
    ap_h = average_precision_score(yht, ph)
    tp = sum(1 for (dn, a, b), pv in zip(mht, ph) if pv > 0.5 and htruth["held"][(a, b)]["edge"])
    fp = sum(1 for (dn, a, b), pv in zip(mht, ph) if pv > 0.5 and not htruth["held"][(a, b)]["edge"])
    fn = int(yht.sum()) - tp
    prec, rec = tp / max(1, tp + fp), tp / max(1, tp + fn)
    print(f"held-out edges: {int(yht.sum())}/{len(yht)} pairs | "
          f"AP={ap_h:.2f} (prevalence {yht.mean():.2f}) | P={prec:.2f} R={rec:.2f}")
    RESULTS["harris"] = dict(ap=float(ap_h), precision=prec, recall=rec,
                             n_pairs=int(len(yht)), n_edges=int(yht.sum()))

fig = render_harris(held, htruth["held"], pred=pred_pairs, fname="harris_held.png")
plt.show()

## 15 · Field report

In [ ]:
mins = (time.time() - RUN_T0) / 60
summary = f"""
=========================================================
 DESIGN ARCHAEOLOGY — field report
=========================================================
 sites excavated ............ {len(all_designs)} designs, {sum(len(d.constraints) for d in all_designs)} constraint artifacts
 oracle cost ................ ≈{n_runs} full place-&-route runs
 vestigial fraction ......... {RESULTS['vestigial_frac']:.0%}
 fossils invisible to lint .. {RESULTS['live_fossil_share']:.0%} of vestigial constraints have LIVE referents

 vestigiality detection (AUROC, held-out design)
   GBT on parser features ... {RESULTS['auroc_held']:.3f}
   dead-referent linter ..... {RESULTS['auroc_linter_held']:.3f}
   (LODO-CV on train sites:   {RESULTS['auroc_cv']:.3f})
 effect-size calibration .... Spearman ρ = {RESULTS['effect_spearman']:.2f}

 deletion experiment (held-out, all-at-once; harm / gain in bands)
   model set (k={RESULTS['deletion']['model top-k']['k']}) ........ harm {RESULTS['deletion']['model top-k']['harm']:.2f} · gain {RESULTS['deletion']['model top-k']['gain']:.2f}
   random set (k={RESULTS['deletion']['random k']['k']}) ....... harm {RESULTS['deletion']['random k']['harm']:.2f} · gain {RESULTS['deletion']['random k']['gain']:.2f}
   linter set (k={RESULTS['deletion']['linter (dead refs)']['k']}) ......... harm {RESULTS['deletion']['linter (dead refs)']['harm']:.2f} · gain {RESULTS['deletion']['linter (dead refs)']['gain']:.2f}
   oracle set (k={RESULTS['deletion']['oracle (upper bound)']['k']}) ....... harm {RESULTS['deletion']['oracle (upper bound)']['harm']:.2f} · gain {RESULTS['deletion']['oracle (upper bound)']['gain']:.2f}

 seriation (held-out) ....... τ = {RESULTS['seriation']['0.0']:.2f} clean, {RESULTS['seriation']['0.4']:.2f} at 40% squashed
 harris matrix .............. AP = {RESULTS.get('harris', {}).get('ap', float('nan')):.2f} on {RESULTS.get('harris', {}).get('n_pairs', 0)} tested pairs
 total wall clock ........... {mins:.1f} min (CPU only)
=========================================================
"""
print(summary)
with open(OUT / "results.json", "w") as f:
    json.dump(RESULTS, f, indent=2, default=float)
print("saved:", OUT / "results.json", "and", len(list(IMG.glob('*'))), "images in", IMG)

## 16 · Reading the field report

Four claims, each with its number above:

- **Fossils are real and linters can't see them.** A majority of deposited constraints are
  vestigial under intervention, and most of those have *live* referents — syntactically
  valid, semantically dead. The dead-referent linter (what industry lint passes do) is at
  chance on the held-out design.
- **The relax-and-re-run oracle can be distilled.** Trees over seventeen parser-level
  features rank fossils far above chance and calibrate to measured effect size — at the
  cost of *one* baseline flow run instead of hundreds.
- **The predictions survive contact with the flow.** Deleting the model's fossil set
  wholesale causes ~no QoR harm and actually *improves* reported timing (the fossils were
  generating noise, not safety). Deleting a same-size random set breaks timing. Note the
  oracle's own set is *not* perfectly safe to delete jointly — individually-vestigial
  artifacts interact, which is precisely why the presupposition DAG matters.
- **Provenance is partially recoverable from content alone.** Seriation recovers a
  substantial fraction of deposition order with timestamps stripped, and is robust to
  squashed commits; interventional Harris edges are predictable well above prevalence.

## 17 · Limitations (the honest trench walls)

- **Synthetic history realism.** The edit grammar is scripted; real revision histories have
  richer sociology (merges, reverts, tool-generated constraints). The corruption ablation
  is a start, not an answer.
- **Small designs, single toy flow.** A few hundred cells and a star-model placer stand in
  for a commercial flow. The mechanism (constraints steer optimization; relaxation moves
  QoR against a noise band) is the same, but absolute numbers will not transfer. Untested
  on commercial tools.
- **Noise-band sensitivity.** Labels depend on the band definition (we use ~2.5σ of the
  Δ-of-means estimator, floored). The full paper ablates the band choice; here it is fixed.
- **Harris-edge sample size.** Interventional pair tests are quadratic; we test a biased
  sample of pairs, so edge recall is measured against a partial truth.

**Why archaeology, really?** The discipline contributes the *label taxonomy*
(use / reuse / residual → active / repurposed / vestigial), the *DAG formalism* (Harris
matrices predate DAG-based build systems by decades), and the *evaluation frame*
(relative dating scored against stratigraphic truth). Everything here is defined
interventionally; the archaeology is load-bearing, not decorative.

## 18 · Toward the full paper

The 8-page version adds: OpenROAD/nangate45 transfer designs (the "real flow" test),
cached code-encoder embeddings of constraint text concatenated to these features, a GNN
over the constraint–object bipartite graph, band-choice ablations, and the
budget: everything above fits in **< 10 GPU-hours** because the GPU only ever sees
final-config training — the oracle is embarrassingly parallel CPU.

*Artifacts of this run:* `results.json`, and every figure as PNG under `images/` in the
notebook's output — including `growth.gif`, the site accumulating its strata.